# Лабораторная работа №4
## Выделение контуров на изображениях

Для свертки использовать написанную ранее в ЛР1 функцию, по необходимость дописать её для работы с масками с четной размерностью.

1. Считать цветное rgb изображение. Преобразовать в градации серого.
2. Написать функцию свёртки (прохождение окном).
3. Сделать выделение контуров методом простого градиента. В качестве значения модуля градиента использовать указанный в вариантах метод.
  *   Вход: изображение из пункта 1
  *   Вывод: бинарное изображение с контурами
4. Сделать выделение контуров методом по вариантам.
  *   Вход: изображение из пункта 1
  *   Вывод: бинарное изображение с контурами

5. Сделать выделение контуров методом с согласованием. Тип функции аппроксимации и размер окна указан по вариантам.
  *   Вход: изображение из пункта 1
  *   Вывод: бинарное изображение с контурами


*Для работы с изображением использовать OpenCV (открытие, сохранение и т.д.). Для визуализации можно использовать matplotlib. Все необходимые для задания функции реализовавать самим, а не использовать готовые в OpenCV, если не указано обратного.*


Ссылки на полезные ресурсы:

1.    [Методичка по локальным методам выделения контуров](https://drive.google.com/file/d/1gjYK-uPsrmzoqXiCxt42t7kGVp3b5cTj/view?usp=drivesdk)
1.    [Методичка по локальным методам выделения контуров 2](https://drive.google.com/file/d/1l0zopB8B4Ep8HyzjQb0YRrLZweYDBtte/view?usp=sharing)
1.    [Документация OpenCV](https://docs.opencv.org/4.x/index.html)


Вариант 3:
1. Модуль градиента аппроксимируется максимумом среди модулей производных
2. Оператор Лапласа
3. Метод с согласованием: аппроксимация поверхностью 1-го порядка, окно 3x3

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

### 1. Считать цветное RGB-изображение. Преобразовать в градации серого.

In [ ]:
def read_rgb_image(image_path: str, size: tuple[int, int] = (255, 255)) -> np.ndarray:
    bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f"Не удалось прочитать файл: {image_path}")
    bgr = cv2.resize(bgr, size, interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


image_path = "image.jpg"
image = read_rgb_image(image_path)
gray_image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image)
axes[0].set_title("RGB-изображение")
axes[0].axis("off")

axes[1].imshow(gray_image, cmap="gray")
axes[1].set_title("Градации серого")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def convolution(image, kernel):
    img = image.astype(np.float64)
    ker = kernel.astype(np.float64)

    kh, kw = ker.shape

    pad_h, pad_w = kh // 2, kw // 2
    ker_rot = np.rot90(ker, 2)  # свёртка = разворот ядра на 180
    padded = np.pad(img, ((pad_h, pad_h), (pad_w, pad_w)), mode="constant")
    out = np.zeros_like(img)

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            patch = padded[i:i + kh, j:j + kw]
            out[i, j] = np.sum(patch * ker_rot)
    return out

def to_binary(image):
    threshold = image.mean()
    return np.where(image > threshold, 255, 0).astype(np.uint8)

### 2. Сделать выделение контуров методом простого градиента.
В качестве значения модуля градиента используется максимум среди модулей производных.

In [ ]:
def simple_gradient_solution(image):
    window_y = np.array([[-1], [1]])
    window_x = np.array([[-1, 1]])

    derivative_y = np.abs(convolution(image, window_y))
    derivative_x = np.abs(convolution(image, window_x))
    derivative = np.maximum(derivative_y, derivative_x)

    return to_binary(derivative)

simple_gradient = simple_gradient_solution(gray_image)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(gray_image, cmap='gray')
plt.title('Исходное изображение')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(simple_gradient, cmap='gray')
plt.title('Метод простого градиента')
plt.axis('off')

plt.show()

### 3. Сделать выделение контуров методом: оператор Лапласа.

In [ ]:
def laplace_solution(image):
    window = np.array([[0, -1, 0],
                       [-1, 4, -1],
                       [0, -1, 0]], dtype=np.float64)

    derivative = np.abs(convolution(image, window))
    return to_binary(derivative)

laplace = laplace_solution(gray_image)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(gray_image, cmap='gray')
plt.title('Исходное изображение')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(laplace, cmap='gray')
plt.title('Оператор Лапласа')
plt.axis('off')

plt.show()

### 4. Сделать выделение контуров методом с согласованием.
Тип функции аппроксимации: поверхность 1-го порядка, размер окна: 3x3.
Это соответствует оператору Превитта.

In [ ]:
def matched_method_solution(image):
    window_y = np.array([[-1, -1, -1],
                         [ 0,  0,  0],
                         [ 1,  1,  1]], dtype=np.float64)

    window_x = np.array([[-1,  0,  1],
                         [-1,  0,  1],
                         [-1,  0,  1]], dtype=np.float64)

    derivative_y = convolution(image, window_y)
    derivative_x = convolution(image, window_x)
    derivative = np.sqrt(derivative_y ** 2 + derivative_x ** 2)

    return to_binary(derivative)

matched_image = matched_method_solution(gray_image)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(gray_image, cmap='gray')
plt.title('Исходное изображение')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(matched_image, cmap='gray')
plt.title('Согласование: поверхность 1-го порядка')
plt.axis('off')

plt.show()